# Disaster Response AI Platform — Full-Dataset GPU Training Pipeline
### End-to-End Model Training, Calibration, Quality Gating & Registry Checkpointing on Google Colab (T4/A100)

This notebook trains the Disaster Response AI multi-stage computer vision models on the **complete, full-scale datasets** using GPU acceleration (`--device cuda`):

1. **Stage 1 Edge Triage**: MobileNetV3-Small on all 6,433+ real aerial UAV images from **AIDER** (Kyrkou et al.).
2. **Stage 2 Structural Damage**: 4-Tier Ordinal Damage CNN on full building crops from **RescueNet** (Bina-Lab Hurricane Ian UAV).
3. **Stage 2 Road Passability**: Binary Accessibility Classifier on full RGB road scenes from **RescueNet**.
4. **Stage 2 Flood Extent U-Net**: 2D Convolutional U-Net on all paired high-resolution UAV images & masks from **FloodNet-Supervised v1.0** (Rahnemoonfar et al.).

**Provenance & Versioning Protocol**: Full-dataset training automatically promotes models to **v2.0.0** in `config/model_registry.json` and updates `models/benchmark_report.json` with single-source-of-truth canonical metrics.

## 1. Hardware & Environment Verification
Verifies NVIDIA GPU availability and CUDA acceleration. Ensure your Colab runtime is set to **GPU** (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "[HARDWARE ERROR] No GPU detected! Please navigate to: "
        "Runtime > Change runtime type > T4 GPU (or A100) before proceeding."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"[OK] Active Compute Device: {gpu_name} ({vram_gb:.1f} GB VRAM)")

## 2. Clone Repository & Install Dependencies
Clones the latest code from GitHub and installs necessary computer vision and evaluation packages.

In [ ]:
# Clone or pull latest repository
!git clone https://github.com/YMP7/Disaster-Response.git || (cd Disaster-Response && git pull)
%cd Disaster-Response

# Install dependencies
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q opencv-python-headless numpy pytest pytest-asyncio anyio

print("[OK] Dependencies installed successfully.")

## 3. Automated Dataset Download & Fail-Loud Verification

This cell downloads and unpacks the 3 disaster benchmark datasets:
- **AIDER**: Via Kaggle API (`clguo1/aiderdata`). Requires `kaggle.json`.
- **FloodNet**: Via Dropbox Direct Archive (`?dl=1`).
- **RescueNet**: Via Dropbox Direct Archive (`?dl=1`).

> **Fail-Loud Safety Guard**: If any dataset is missing, truncated, or below expected file count thresholds, this cell **aborts immediately** with clear instructions rather than allowing training to run on incomplete data.

In [ ]:
import os
import sys
from pathlib import Path
import subprocess

data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# A. AIDER Dataset Download (~2.5 GB)
# ---------------------------------------------------------------------
aider_dir = data_dir / "AIDER"
aider_files = list(aider_dir.rglob("*.jpg")) if aider_dir.exists() else []

if len(aider_files) < 5000:
    print("\n=== [1/3] Downloading AIDER Dataset from Kaggle ===")
    kaggle_cfg = Path.home() / ".kaggle" / "kaggle.json"
    if not kaggle_cfg.exists():
        # Prompt user to upload kaggle.json if missing
        try:
            from google.colab import files
            print("kaggle.json not found. Please upload your Kaggle API key (kaggle.json):")
            uploaded = files.upload()
            if "kaggle.json" in uploaded:
                (Path.home() / ".kaggle").mkdir(parents=True, exist_ok=True)
                with open(kaggle_cfg, "wb") as f:
                    f.write(uploaded["kaggle.json"])
                os.chmod(kaggle_cfg, 0o600)
                print("[OK] kaggle.json configured.")
        except Exception as e:
            print(f"Note: {e}")

    if not kaggle_cfg.exists():
        raise RuntimeError(
            "\n" + "!" * 80 + "\n"
            "[DATASET ERROR] AIDER download requires a Kaggle API key (kaggle.json).\n"
            "To get one:\n"
            "  1. Go to https://www.kaggle.com/settings -> Click 'Create New Token'.\n"
            "  2. Upload kaggle.json to this Colab session using the Files sidebar or files.upload().\n"
            "  3. Move it: !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json\n"
            "Alternatively, download manually from: https://www.kaggle.com/datasets/clguo1/aiderdata\n"
            "and unzip into 'data/AIDER/'.\n"
            + "!" * 80
        )

    !pip install -q kaggle
    !kaggle datasets download -d clguo1/aiderdata -p /tmp/aider
    !mkdir -p data/AIDER && unzip -q /tmp/aider/*.zip -d data/AIDER/
    !rm -rf /tmp/aider
else:
    print(f"[OK] AIDER dataset already present ({len(aider_files)} images).")


# ---------------------------------------------------------------------
# B. FloodNet Dataset Download (~5.2 GB)
# ---------------------------------------------------------------------
flood_dir = data_dir / "FloodNet"
flood_files = list(flood_dir.rglob("*.jpg")) if flood_dir.exists() else []

if len(flood_files) < 1500:
    print("\n=== [2/3] Downloading FloodNet-Supervised v1.0 from Dropbox Archive ===")
    flood_url = "https://www.dropbox.com/scl/fo/k33qdif15ns2qv2jdxvhx/ANGaa8iPRhvlrvcKXjnmNRc?rlkey=ao2493wzl1cltonowjdbrnp7f&e=5&dl=1"
    !mkdir -p /tmp/floodnet
    !curl -L -o /tmp/floodnet/floodnet.zip "$flood_url"
    
    # Verify downloaded file is a valid zip (not an HTML error page)
    if not os.path.exists("/tmp/floodnet/floodnet.zip") or os.path.getsize("/tmp/floodnet/floodnet.zip") < 1000000:
        raise RuntimeError(
            "\n" + "!" * 80 + "\n"
            "[DATASET ERROR] FloodNet Dropbox link returned an invalid or rate-limited archive.\n"
            "Please download FloodNet manually from:\n"
            "https://www.dropbox.com/scl/fo/k33qdif15ns2qv2jdxvhx/ANGaa8iPRhvlrvcKXjnmNRc?rlkey=ao2493wzl1cltonowjdbrnp7f&e=5&dl=0\n"
            "and upload/unzip into 'data/FloodNet/'.\n"
            + "!" * 80
        )
    
    !mkdir -p data/FloodNet && unzip -q /tmp/floodnet/floodnet.zip -d data/FloodNet/
    !rm -rf /tmp/floodnet
else:
    print(f"[OK] FloodNet dataset already present ({len(flood_files)} images).")


# ---------------------------------------------------------------------
# C. RescueNet Dataset Download (~10.4 GB)
# ---------------------------------------------------------------------
rescue_dir = data_dir / "RescueNet"
rescue_files = list(rescue_dir.rglob("*.png")) if rescue_dir.exists() else []

if len(rescue_files) < 2000:
    print("\n=== [3/3] Downloading RescueNet Post-Hurricane Ian UAV Dataset ===")
    rescue_url = "https://www.dropbox.com/scl/fo/ntgeyhxe2mzd2wuh7he7x/AHJ-cNzQL-Eu04HS6bvBgcw?rlkey=6vxiaqve9gp6vzvzh3t5mz0vv&e=6&dl=1"
    !mkdir -p /tmp/rescuenet
    !curl -L -o /tmp/rescuenet/rescuenet.zip "$rescue_url"
    
    if not os.path.exists("/tmp/rescuenet/rescuenet.zip") or os.path.getsize("/tmp/rescuenet/rescuenet.zip") < 1000000:
        raise RuntimeError(
            "\n" + "!" * 80 + "\n"
            "[DATASET ERROR] RescueNet Dropbox link returned an invalid or rate-limited archive.\n"
            "Please download RescueNet manually from:\n"
            "https://www.dropbox.com/scl/fo/ntgeyhxe2mzd2wuh7he7x/AHJ-cNzQL-Eu04HS6bvBgcw?rlkey=6vxiaqve9gp6vzvzh3t5mz0vv&e=6&dl=0\n"
            "and upload/unzip into 'data/RescueNet/'.\n"
            + "!" * 80
        )
    
    !mkdir -p data/RescueNet && unzip -q /tmp/rescuenet/rescuenet.zip -d data/RescueNet/
    !rm -rf /tmp/rescuenet
else:
    print(f"[OK] RescueNet dataset already present ({len(rescue_files)} masks).")


# ---------------------------------------------------------------------
# D. Strict Dataset Audit & Threshold Verification
# ---------------------------------------------------------------------
final_aider = len(list(aider_dir.rglob("*.*")))
final_flood = len(list(flood_dir.rglob("*.*")))
final_rescue = len(list(rescue_dir.rglob("*.*")))

print("\n" + "=" * 60)
print("DATASET VERIFICATION AUDIT REPORT")
print("=" * 60)
print(f"AIDER Files:     {final_aider} (Expected >= 5,000)")
print(f"FloodNet Files:  {final_flood} (Expected >= 5,000)")
print(f"RescueNet Files: {final_rescue} (Expected >= 10,000)")

assert final_aider >= 5000, f"AIDER file count ({final_aider}) is below expected full dataset threshold!"
assert final_flood >= 5000, f"FloodNet file count ({final_flood}) is below expected full dataset threshold!"
assert final_rescue >= 10000, f"RescueNet file count ({final_rescue}) is below expected full dataset threshold!"
print("\n[PASSED] All datasets verified complete and ready for full-scale GPU training.")

## 4. Full-Dataset GPU Training Execution

Executes `models/train.py` with the new flags:
- `--full-dataset`: Removes all sample caps (loads all 6,433 AIDER, all RescueNet crops & scenes, and all FloodNet masks).
- `--device cuda`: Runs all model forward & backward passes on the GPU.
- `--batch-size 32`: Standardized batch size for GPU parallelization.
- Quality gates: Road passability classifier must outperform chance level (> 60%) to be marked active.

In [ ]:
!python models/train.py \
    --full-dataset \
    --device cuda \
    --batch-size 32 \
    --epochs-s1 3 \
    --epochs-s2 8 \
    --epochs-flood 5

## 5. Test Suite & Cross-File Canonical Consistency Verification

Runs the entire 27-test automated test suite directly on the newly trained v2.0.0 checkpoints to verify:
- Zero regression across all modules.
- Checkpoint isolation and clean initialization.
- Canonical consistency across `benchmark_report.json` and `model_registry.json`.
- Road passability quality gate enforcement.

In [ ]:
!pytest -v tests/

import json
with open("config/model_registry.json") as f:
    registry = json.load(f)

print("\nActive Models in Registry:")
for stage, model_id in registry.get("active_models", {}).items():
    meta = registry["models"][model_id]
    print(f"  - {stage}: {model_id} (version: {meta['version']}, status: {meta['status']})")

with open("models/benchmark_report.json") as f:
    report = json.load(f)
print(f"\nFull Training Elapsed Time: {report.get('elapsed_seconds', 0.0):.1f} seconds")

## 6. Packaging & Exporting Trained Artifacts

Bundles the trained `.pt` model weights, `model_registry.json`, and `benchmark_report.json` into a deployable zip archive for seamless download and integration back into your local repository or production service.

In [ ]:
!zip -r disaster_response_v2_artifacts.zip \
    models/weights/*.pt \
    config/model_registry.json \
    models/benchmark_report.json

try:
    from google.colab import files
    files.download("disaster_response_v2_artifacts.zip")
    print("[OK] Download initiated for disaster_response_v2_artifacts.zip")
except Exception as e:
    print(f"Artifact zip created at disaster_response_v2_artifacts.zip: {e}")